# Stokes pressure: archived convergence and interface jumps

This notebook displays archived **pyMHM** USFEM P1/P1 and Taylor–Hood P2/P1
results, their recorded full-system checks, and independent globally conforming
**DOLFINx/UFL** references. The pressure may jump across MHM macrofaces.
Neither a small algebraic residual nor a visually smooth velocity establishes
pressure accuracy.

The exact data follow [Araya et al. (2017)](https://doi.org/10.1016/j.cma.2017.05.027).
These low-order local spaces differ from the paper's higher-order experiments;
the figures here compare computed formulations with analytical fields, rather
than digitized curves from the article. This notebook only reads previously
recorded values and figures, with no solve or comparison recomputation.

In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))


In [ ]:
from pathlib import Path
import json
from IPython.display import Image, JSON, display

data = json.loads((root / "examples/results/flow-audit.json").read_text())

display(JSON({
    "problem": data["problem"],
    "code": "pyMHM",
    "recorded_full_system_checks": [
        {key: value for key, value in row.items() if key != "profile"}
        for row in data["H_refinement"]
        if "monolithic_relative_field_difference" in row
    ],
}))

## Separate global, local, and trace errors

The archived sweep contains five macro resolutions and five local resolutions.
The **DOLFINx/UFL** references use the same microtriangles and the corresponding
USFEM P1/P1 or Taylor–Hood P2/P1 formulation, with strong velocity boundary data
and globally continuous pressure. They are independent conforming comparisons,
not the same discrete MHM problem. The displayed full-system checks were
recorded for the MHM system at macro resolution n=4.

In [ ]:
for method in ("usfem", "taylor-hood"):
    display(JSON({
        "pyMHM_formulation": method,
        "recorded_macro_refinement": [
            {key: row[key] for key in ("n", "velocity_l2", "pressure_l2", "divergence_l2")}
            for row in data["H_refinement"] if row["formulation"] == method
        ],
        "DOLFINx_UFL_conforming_reference": [
            row for row in data["conforming_reference"] if row["formulation"] == method
        ],
    }))
display(Image(filename=str(root / "docs/figures/flow-audit/convergence.png")))
display(Image(filename=str(root / "docs/figures/flow-audit/trace-local.png")))

## Both sides of every pressure discontinuity

The plot evaluates the actual local polynomials and ends each segment at its macrocell boundary. Open endpoints expose the two traces; no averaging or smoothing removes their difference. The lower panels use different vertical scales. The refined Taylor–Hood pressure error is about 0.0398% in L2. Low-order USFEM still has localized pressure errors, despite a much better overall error than the coarse case.

In [ ]:
display(Image(filename=str(root / "docs/figures/flow-audit/pressure-profiles.png")))